## RAG approach
1. Dense/vector search for semantic similarity
2. BM25/keyword search for exact medical terms
3. Metadata filtering for trusted sources/topic/date
4. Reranking to choose the strongest evidence
5. Evidence-grounded generation where the LLM is explicitly restricted to retrieved material

Original query: “I saw on TikTok that taking 10 mg of melatonin every night fixes sleep deprivation. Is that true?”
       ↓
Claim extraction
       ↓
"10 mg melatonin every night fixes sleep deprivation"
       ↓
Query rewriting
       ↓
1. melatonin effectiveness for sleep deprivation
2. melatonin dosage young adults
3. melatonin long-term safety
       ↓
Hybrid retrieval
   ┌───────────────┐
   │ Dense search  │
   │ BM25 search   │
   └──────┬────────┘
          ↓
Trusted-source filter
          ↓
Reranker
          ↓
Top 3–5 evidence chunks
          ↓
Evidence sufficiency check
          ↓
LLM
          ↓
Grounded educational response + citations

## Overall structure for the project.

                    USER
                      │
                      ▼
             Text / Image Input
                      │
                      ▼
              INPUT GUARDRAIL
         privacy / safety / scope
                      │
                      ▼
              CLAIM EXTRACTION
                      │
                      ▼
           QUERY CLASSIFICATION
                     /              
                Supplements          
                    |       
                    │
                    ▼
              QUERY REWRITING
                    │
                    ▼
        ┌───────────────────────┐
        │      HYBRID RAG       │
        │                       │
        │ Dense vector search   │
        │          +            │
        │ BM25 keyword search   │
        └───────────┬───────────┘
                    │
                    ▼
           SOURCE/METADATA FILTER
                    │
                    ▼
                RERANKER
                    │
                    ▼
          EVIDENCE SUFFICIENT?
              /           \
            No             Yes
            │               │
      Rewrite/search        ▼
         again       EVIDENCE PACKAGE
                            │
                            ▼
                           LLM
                            │
                            ▼
                    OUTPUT GUARDRAIL
                   /        |        \
             grounded?   safe?    citations?
                            │
                            ▼
                  STRUCTURED RESPONSE
                  • claim assessment
                  • explanation
                  • evidence strength
                  • uncertainty
                  • sources
                            │
                            ▼
                           USER

## RAG technique:
Use Corrective RAG.

Retrieve evidence
       ↓
Evaluate retrieval quality
       ↓
Is the evidence good?
   /            \
 Yes            No
 │               │
Use it      Rewrite query
 │               │
 │          Retrieve again
 │               │
 └───────┬───────┘
         ▼
     Generate

## Imports + Model prepping

In [51]:
import os
from pathlib import Path
import pandas as pd
import chromadb
from openai import OpenAI
from sentence_transformers import SentenceTransformer

In [52]:
client = OpenAI(
    api_key=os.environ["COMP3110_API_KEY"],
    base_url=os.environ["COMP3110_MODEL_BASE_URL"]
)

MODEL = "aws-nova-pro-v1"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4330.22it/s]


## Grab Documents


In [53]:
# TODO grab at least 10 trusted documents about supplements
# Try to do each document in this structure: id, title, source, url, topic, text.

# documents = [
#     {
#         "id": "",
#         "title": "",
#         "source": "",
#         "url": "",
#         "topic": "",
#         "text": ""
#     },
# ]


documents = [
    {
        "id": "nih_creatine_performance",
        "title": "Dietary Supplements for Exercise and Athletic Performance",
        "source": "NIH Office of Dietary Supplements",
        "url": "https://ods.od.nih.gov/factsheets/ExerciseAndAthleticPerformance-Consumer/",
        "topic": "creatine",
        "text": """
# Creatine
Creatine is a compound that is stored in your muscles and supplies them with energy. Your body produces some creatine (about 1 gram a day), and you get some creatine from eating animal-based foods, such as beef and salmon (about 500 milligrams in a 4-ounce serving). However, it is only when you take much larger amounts of creatine from dietary supplements that it might improve certain types of performance.	

## Does it work?
Creatine supplements can increase strength, power, and the ability to contract muscles for maximum effort, but the extent of performance improvements from creatine supplements differs among individuals.

Use of creatine supplements for several weeks or months can help with training. Overall, creatine enhances performance during repeated short bursts of intense, intermittent activity (lasting up to about 2.5 minutes at a time), such as sprinting and weightlifting. Creatine seems to have little value for endurance activities, such as distance running, cycling, or swimming.	

## Is it safe?
Creatine is safe for healthy adults to take for several weeks or months. It also seems safe for long-term use over several years. Creatine usually causes some weight gain because it increases water retention. Rare individual reactions to creatine include some muscle stiffness and cramps as well as GI distress.	

## Bottom Line
Sports-medicine experts agree that creatine supplements can improve performance in activities that involve intense effort followed by short recovery periods. It can also be valuable in training for certain athletic competitions. In studies, people often took a loading dose of about 20 grams per day of creatine (in four equal portions) for 5 to 7 days and then 3 to 5 grams a day. Creatine monohydrate is the most widely used and studied form of creatine in supplements.
        """
    },
    {
        "id": "nih_creatine_dosing",
        "title": "Exercise and Athletic Performance - Health Professional",
        "source": "NIH Office of Dietary Supplements",
        "url": "https://ods.od.nih.gov/factsheets/ExerciseandAthleticPerformance-HealthProfessional/",
        "topic": "creatine",
        "text": """
# Creatine
Creatine is one of the most thoroughly studied and widely used dietary supplements to enhance exercise and sports performance. Creatine is produced endogenously and obtained from the diet in small amounts. It helps generate ATP and thereby supplies the muscles with energy, particularly for short-term events. Creatine might improve muscle performance in four ways: by increasing stores of phosphocreatine used to generate ATP at the beginning of intense exercise, accelerating the re-synthesis of phosphocreatine after exercise, depressing the degradation of adenine nucleotides and the accumulation of lactate, and/or enhancing glycogen storage in skeletal muscles.

The liver and kidneys synthesize about 1 g/day creatine from the amino acids glycine, arginine, and methionine. Animal-based foods, such as beef (2 g/lb), pork (2.3 g/lb), and salmon (2 g/lb), also contain creatine. A person weighing 154 pounds has about 120 g creatine and phosphocreatine in his or her body, almost all in the skeletal and cardiac muscles. However, it is only when users consume much greater amounts of creatine over time as a dietary supplement that it could have ergogenic effects. Metabolized creatine is converted into the waste product creatinine, which is eliminated from the body through the kidneys.

## Efficacy
Studies in both laboratory and sports settings have found that short-term creatine supplementation (for 5 to 7 days) in both men and women often significantly increases strength (e.g., for bench presses) and power (e.g., for cycling), work involving multiple sets of maximal effort muscle contractions, and sprinting and soccer performance. In one example, a study randomized 14 healthy, resistance-trained men (age 19–29 years) to receive 25 g creatine monohydrate or a placebo for 6–7 days. Participants taking the supplement had significant improvements in peak power output during all five sets of jump squats and in repetitions during all five sets of bench presses on three occasions. In another study, 18 well-trained male sprinters age 18–24 years received either 20 g/day creatine or a placebo for 5 days. Compared with those taking the placebo, participants taking the creatine improved their performance in both 100-meter sprints and six intermittent 60-m sprints.

Supplementation with creatine over weeks or months helps training adaptations to structured, increased workloads over time. For example, in a randomized study of 14 female collegiate soccer players during the off-season, those who received creatine (15 g/day for 1 week and then 5 g/day for 12 weeks) had significantly greater increases in muscle strength, as measured by bench press and full-squat maximal strength testing, but not lean tissue compared with participants who took a placebo.

Individuals have varied responses to creatine supplementation, based on factors such as diet and the relative percentages of various muscle fiber types. Vegetarians, for example, with their lower muscle creatine content, might have greater responses to supplementation than meat eaters. Overall, creatine enhances performance during repeated short bursts of high-intensity, intermittent activity, such as sprinting and weight lifting, where energy for this predominantly anaerobic exercise comes mainly from the ATP-creatine phosphate energy system.

Creatine supplementation seems to be of little value for endurance sports, such as distance running or swimming, that do not depend on the short-term ATP-creatine phosphate system to provide short-term energy, and it leads to weight gain that might impede performance in such sports. Furthermore, in predominantly aerobic exercise lasting more than 150 seconds, the body relies on oxidative phosphorylation as the primary energy source, a metabolic pathway that does not require creatine.

## Safety
Studies have found no consistent set of side effects from creatine use, except that it often leads to weight gain, because it increases water retention and possibly stimulates muscle protein synthesis. Several studies have found that supplemental creatine monohydrate, when used for a strength-training program, can lead to a 1–2 kg increase in total body weight in a month.

Creatine is considered safe for short-term use by healthy adults. In addition, evidence shows that use of the product for several years is safe. Anecdotal reactions to creatine use include nausea, diarrhea and related gastrointestinal distress, muscle cramps, and heat intolerance. Creatine supplementation may reduce the range of motion of various parts of the body (such as the shoulders, ankles, and lower legs) and lead to muscle stiffness and resistance to stretching. Adequate hydration while taking creatine might minimize these uncommon risks.

## Implications for use
In a position statement, the AND, DoC, and ACSM advise that creatine enhances performance of cycles of high-intensity exercise followed by short recovery periods and improves training capacity. In its position statement, the ISSN states that creatine monohydrate is the most effective nutritional supplement currently available for enhancing capacity for high-intensity exercise and lean body mass during exercise. The ISSN contends that athletes who supplement with creatine have a lower incidence of injuries and exercise-related side effects compared to those who do not take creatine. The Australian Institute of Sport supports the use of creatine for improving sports performance in suitable athletic competitions under the direction of an expert in sports medicine, but it notes that more research might be required to understand how the supplement should be used for best results.

A typical protocol for creatine supplementation in adults, regardless of sex or body size, consists of a loading phase for 5–7 days, when users consume 20 g/day creatine monohydrate in four portions of 5 g, followed by a maintenance phase of 3–5 g/day. In some studies, the loading dose is based on body weight (e.g., 0.3 g/kg). Another creatine supplementation protocol consists of taking single doses of about 3–6 g/day (0.03–0.1 g/kg body weight) for 3 to 4 weeks, without a loading phase, to produce ergogenic effects.

Creatine monohydrate, which is 88% creatine by weight, is the most widely used and studied form. Other, usually more expensive, forms of creatine (e.g., creatine ethyl ester, creatine alpha-ketoglutarate, and buffered forms of creatine) have not been proven to have superior ability to creatine monohydrate for enhancing muscle creatine levels, digestibility, product stability, or safety.
        """
    },
    {
        "id": "nih_creatine_safety",
        "title": "Bodybuilding and Performance Enhancement Supplements",
        "source": "NIH NCCIH",
        "url": "https://www.nccih.nih.gov/health/bodybuilding-and-performance-enhancement-supplements",
        "topic": "creatine",
        "text": """
# What is creatine?
Creatine is an amino acid produced by the body and a popular dietary supplement. It may somewhat enhance the effects of exercise on strength, muscle mass, and endurance. But creatine can have some short-term side effects, and its long-term effects haven’t been well studied.

# Is creatine safe?
There have been reports that creatine may impair liver and kidney function. Creatine has also been linked to an increased risk of compartment syndrome, a condition where pressure builds in a muscle compartment and prevents blood flow. People at risk of kidney problems should check with their health care providers before using creatine and be carefully monitored while using it.

There are no data documenting the safety of creatine in children or adolescents. The American Academy of Pediatrics and the American College of Sports Medicine warn that teens should not use performance-enhancing supplements, including creatine, because of the possible health risks.
        """
    },
    {
        "id": "ais_creatine_performance",
        "title": "Creatine",
        "source": "Australian Institute of Sport",
        "url": "https://www.ausport.gov.au/ais/nutrition/supplements/group_a/performance-supplements2/creatine",
        "topic": "creatine",
        "text": """
# Creatine Monohydrate

## What is creatine?
Creatine is a non-essential nutrient that is endogenously synthesized (about 1 g/d) and also ingested through the diet (about 1 g/d). Creatine is heavily concentrated in animal muscle (e.g. meat and fish), which is the primary dietary source for omnivores. Muscle and blood creatine levels are reduced in vegetarians who consume little in the diet.

Most creatine is stored in skeletal muscle where it exists in free (i.e. creatine) and phosphorylated (i.e. phosphorylcreatine) forms. Creatine and phosphorylcreatine provide energy to support brief, intense exercise. Although the rate of energy production from muscle creatine is very high, storage capacity is very limited. There is enough creatine in skeletal muscle to support about 8 to 10 seconds of maximal exercise. Creatine monohydrate is a dietary supplement that, when ingested in accordance with current guidelines, can increase skeletal muscle creatine and phosphorylcreatine and subsequently improve high-intensity exercise performance.

A small amount of creatine is present in the brain, where it is also used to support energy production. Brain creatine can be increased with creatine monohydrate supplementation, albeit to a smaller amount than the increase seen in skeletal muscle. The benefits of creatine supplementation on brain health include improved cognitive processing and potentially reduced damage and enhanced recovery from mild traumatic brain injury (mTBI/concussion).

## What does creatine monohydrate look like?
Creatine monohydrate is a white powder that can be ingested after combining it with liquid (e.g. a post-exercise protein-carbohydrate recovery drink) or food (e.g. Greek yogurt). Creatine supplements should be ingested immediately following mixing, as in liquid form, creatine quickly degrades to creatinine.

Creatine monohydrate is very well absorbed (>99%) whereas alternate creatine supplements (e.g. creatine ethyl ester) that advertise “better absorption” do not have data to back such claims. Multiple creatine dietary supplements are available for purchase, but about 99% of the safety and efficacy data are available on creatine monohydrate powder. There is no scientific reason to take a creatine supplement other than creatine monohydrate.

## How and when is creatine used?
Professor Roger Harris first demonstrated that oral ingestion of creatine monohydrate can increase muscle creatine and phosphorylcreatine. After many years of study, multiple research teams confirm that muscle creatine can be increased by ingesting about 5 grams of creatine monohydrate, 4 times per day, for about 5 days (i.e. “creatine loading”). Suggested dosing based on body mass is about 0.3 g of creatine/kg body mass per day, for 5 days, typically in 3-4 divided doses (i.e. with meals), followed by a maintenance dose of 0.03g/ kg body mass once a day thereafter.

Subsequently, Professor Eric Hultman showed that increased muscle creatine could be maintained after creatine loading with ingestion of a maintenance dosage of about 3 to 5 g/d. Alternatively, one could bypass the loading phase, simply ingest a maintenance dose (3 to 5 g/d, and increase muscle creatine to saturation levels over about 4 weeks.

Muscle creatine uptake is insulin mediated, so a larger increase in muscle creatine in response to supplementation could be obtained with coingestion of a meal that generates an acute increase in blood insulin levels. Early studies used large amounts of simple sugars to accomplish this (i.e. >90 g of sugar + 5 g creatine four times/d), but subsequent studies confirm the same effect can be accomplished by ingesting creatine following a meal that includes both protein (50 g) and carbohydrate (50 g) rich foods.

Muscle creatine uptake is similarly increased when supplementation is combined with exercise which has insulin like effects. Although there are not many data to demonstrate that post-exercise creatine ingestion is more effective than pre-exercise ingestion, ingestion of creatine supplements following exercise and with the post-exercise meal is prudent advice and may help athletes establish a habit of proper postexercise nutritional intake.

Creatine loading is analogous to carbohydrate loading. Physical activities, such as endurance exercise, that are limited by carbohydrate availability and metabolism may benefit from carbohydrate loading (i.e. several days of a high-carbohydrate diet). Physical activities, such as sprinting, that are limited by creatine availability and metabolism may benefit from creatine loading/supplementation.

Individuals with the lowest muscle creatine (e.g. vegetarians) have the largest potential for increase in response to supplementation. Muscle creatine levels appear to be relatively unaffected by training style or intensity (i.e. sprinters do not necessarily have high muscle creatine and sprint training does not increase muscle creatine). In response to increased (e.g. supplementation) or decreased (e.g. changing to a meat free diet) dietary creatine intake, muscle creatine quickly increases or decreases, respectively.

It is consistently found that creatine supplementation combined with resistance exercise improves resistance training outcomes, such as muscle strength, endurance, and muscle hypertrophy. This points to creatine supplementation being an effective training aid to augment strength and conditioning programs. See the performance effects section below.

Creatine supplementation improves the performance of brief (usually <30 sec), high-intensity exercise, especially when there are repeated bouts. These are common characteristics to many team sports, indicating that creatine supplementation can improve sports performance across a wide range of sports and activities. Maximal exercise performance is also enhanced when sprints are included during and/or at the end of endurance exercise events. See the performance effects section below.

There is some indication that creatine supplementation can improve recovery from periods of disuse atrophy, such as when recovering from an injury. Extremely low levels of physical activity, such as during immobilization, result in decreased muscle creatine, strength, endurance, and mass, among many other adverse changes, while creatine supplementation attenuates or reverses these decrements. See the performance effects section below.

Creatine supplementation has multiple direct effects on muscle (e.g. increased glycogen, phosphorylcreatine resynthesis, growth factor expression, satellite cell number, cellular hydration, etc.) which could indirectly benefit athletic performance, adaptation to exercise training, or muscular performance in a number of different patient populations. See the performance effects section below.

## Reported effects of creatine monohydrate
The AIS document reports the following effects in its table of known supplementation effects:
- High-intensity laboratory exercise (maximal, under 30 seconds): Increased peak and mean power, and reduced fatigue, especially across repeated efforts.
- High-intensity field exercise (maximal, under 30 seconds): Increased speed and reduced time to cover a fixed distance.
- Swimming: Increased power and improved times; benefits are more likely during repeated sprint intervals than a single sprint.
- Resistance exercise: Increased lifting volume, repetitions at a given weight, strength, and lean body mass.
- Sprinting during or after endurance exercise: Increased power and speed, with reduced fatigue and completion time.
- Rehabilitation after extreme inactivity: Better maintenance of muscle mass, strength and endurance, alongside changes in muscle creatine, GLUT, glycogen and growth factor expression.
- Brain health and concussion: The AIS document describes potential improvements in cognitive processing under energy stress and possible benefits following mild traumatic brain injury. These statements reflect the source document and should not be treated as established clinical treatment recommendations.
There is no evidence of systematic serious adverse effects related to creatine monohydrate supplementation. Speculation and anecdotes about muscle, renal, and thermoregulatory dysfunction are not supported with research or post-marketing surveillance., 13 However, there are some implications of creatine supplementation that warrant discussion, including acute weight gain and gastrointestinal tract distress.

## Rapid weight gain
As carbohydrate ingestion and increased muscle glycogen is associated with an acute increase in body mass secondary to increased body water, increased creatine ingestion and muscle creatine is also associated with weight gain/increased body water. However, with creatine supplements, this increase in body mass is maintained as long as muscle creatine remains elevated. Following cessation of creatine supplementation, muscle creatine levels, and subsequently body mass, decrease slowly to normal over 4 to 6 weeks’ time. Although this may only be a maximum of 1 or 2 kg, this could be problematic for athletes attempting to “make weight”.

Creatine supplementation offers a metabolic advantage but could present a biomechanical disadvantage for some athletes. In theory, body weight supported sports (e.g. running) could be negatively impacted by creatine supplementation. It appears these concerns are unfounded, as studies have showed improved running and swimming performance, but weight gain in some sports such as pole vaulting, in theory, could present a challenge.

## Gastrointestinal discomfort
Some people may experience mild, temporary gastrointestinal (GI) upset during supplementation, although this is anecdotal, and not widely reported in the literature.

Avoiding the loading phase in favour of the lower-dose, longer-duration supplementation protocol, ingesting creatine with meals, not ingesting creatine at the same time as high-fibre foods or supplements that are known to increase GI disturbances (e.g. sodium bicarbonate) are all sensible decisions to help avoid GI upset.

As with any dietary supplement, experimentation should be conducted in the off-season.

## Supplement safety and anti-doping considerations
The AIS advises athletes to consult an accredited sports dietitian and follow their sporting organisation’s supplement policies. Supplements can carry anti-doping risks; even batch-tested products cannot guarantee the absence of prohibited substances.
        """
    },
    {
        "id": "ais_creatine_concerns",
        "title": "Creatine - Concerns and Considerations",
        "source": "Australian Institute of Sport",
        "url": "https://www.ausport.gov.au/ais/nutrition/supplements/group_a/performance-supplements2/creatine/are-there-any-concerns-or-considerations",
        "topic": "creatine",
        "text": """
# Are there any concerns or considerations?

There is no evidence of systematic serious adverse effects related to creatine monohydrate supplementation. Speculation and anecdotes about muscle, renal, and thermoregulatory dysfunction are not supported with research or post-marketing surveillance. However, there are some implications of creatine supplementation that warrant discussion, including acute weight gain and gastrointestinal tract distress.

## Rapid weight gain
- As carbohydrate ingestion and increased muscle glycogen is associated with an acute increase in body mass secondary to increased body water, increased creatine ingestion and muscle creatine is also associated with weight gain/increased body water. However, with creatine supplements, this increase in body mass is maintained as long as muscle creatine remains elevated. Following cessation of creatine supplementation, muscle creatine levels, and subsequently body mass, decrease slowly to normal over 4 to 6 weeks’ time. Although this may only be a maximum of 1 or 2 kg, this could be problematic for athletes attempting to “make weight”.
- Creatine supplementation offers a metabolic advantage but could present a biomechanical disadvantage for some athletes. In theory, body weight supported sports (e.g. running) could be negatively impacted by creatine supplementation. It appears these concerns are unfounded, as studies have showed improved running and swimming performance, but weight gain in some sports such as pole vaulting, in theory, could present a challenge.

## Gastointestinal tract distress
- Some people may experience mild, temporary gastrointestinal (GI) upset during supplementation, although this is anecdotal, and not widely reported in the literature.
- Avoiding the loading phase in favour of the lower-dose, longer-duration supplementation protocol, ingesting creatine with meals, not ingesting creatine at the same time as high-fibre foods or supplements that are known to increase GI disturbances (e.g. sodium bicarbonate) are all sensible decisions to help avoid GI upset.
- As with any dietary supplement, experimentation should be conducted in the off-season.
        """
    },
]


In [54]:
documents_df = pd.DataFrame(documents)
documents_df[["id", "title", "source", "topic"]]

,id,title,source,topic
0,nih_creatine_performance,Dietary Supplements for Exercise and Athletic ...,NIH Office of Dietary Supplements,creatine
1,nih_creatine_dosing,Exercise and Athletic Performance - Health Pro...,NIH Office of Dietary Supplements,creatine
2,nih_creatine_safety,Bodybuilding and Performance Enhancement Suppl...,NIH NCCIH,creatine
3,ais_creatine_performance,Creatine,Australian Institute of Sport,creatine
4,ais_creatine_concerns,Creatine - Concerns and Considerations,Australian Institute of Sport,creatine


# Chunk Documents

### What overlap means
Overlapping is repeating a small portion (50 words) from the end of one chunk to the start of the next

### What does this do?
When a document is split in chunks, no sentence, idea or phrases are cut awkwardly when a document is divided. It also preserves context and allows the system to piece information together better.


In [55]:
def chunk_text(
    text, 
    chunk_size=200,
    overlap=30
):
    chunks = []
    words = text.split()
    start = 0

    while start < len(words):
        chunk = words[start:start + chunk_size]
        chunks.append(" ".join(chunk))
        start += chunk_size - overlap
    return chunks

In [56]:
chunks = []

for document in documents:

    document_chunks = chunk_text(
        document["text"]
    )

    for index, text in enumerate(
        document_chunks
    ):

        chunks.append(
            {
                "id": f"{document['id']}_{index}",
                "text": text,
                "title": document["title"],
                "source": document["source"],
                "url": document["url"],
                "topic": document["topic"],
            }
        )

In [57]:
chunks_df = pd.DataFrame(chunks)
chunks_df.head()

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Documents: 5
Chunks: 20


## Embeddings


In [58]:
def create_embedding(text: str) -> list[float]:
    embedding = embedding_model.encode(text)
    return embedding.tolist()

In [59]:
# Test embedding
test_embedding = create_embedding("Does creatine improve muscle strength?")

print(len(test_embedding))
print(test_embedding[:5])

384
[-0.026150330901145935, 0.005148985423147678, -0.027785545215010643, -0.0001068856508936733, -0.06546502560377121]


## Vector Database

In [60]:
chroma_client = chromadb.Client()

if "medical_evidence" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("medical_evidence")

collection = chroma_client.get_or_create_collection(name="medical_evidence")

## Embed and store the chunks

In [61]:
for chunk in chunks:

    embedding = create_embedding(chunk["text"])

    collection.add(
        ids=[chunk["id"]],
        documents=[chunk["text"]],
        embeddings=[embedding],
        metadatas=[
            {
                "title": chunk["title"],
                "source": chunk["source"],
                "url": chunk["url"],
                "topic": chunk["topic"],
            }
        ],
    )

# Check collection
collection.count()

20

## Testing Vector Retrieval

In [62]:
def vector_search(query, top_k=5,):
    query_embedding = create_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )

    return results

# Testing
results = vector_search("Can magnesium help me sleep?")
results

{'ids': [['ais_creatine_performance_6',
   'nih_creatine_dosing_3',
   'ais_creatine_performance_1',
   'ais_creatine_performance_5',
   'ais_creatine_concerns_1']],
 'embeddings': None,
 'documents': [['seconds): Increased speed and reduced time to cover a fixed distance. - Swimming: Increased power and improved times; benefits are more likely during repeated sprint intervals than a single sprint. - Resistance exercise: Increased lifting volume, repetitions at a given weight, strength, and lean body mass. - Sprinting during or after endurance exercise: Increased power and speed, with reduced fatigue and completion time. - Rehabilitation after extreme inactivity: Better maintenance of muscle mass, strength and endurance, alongside changes in muscle creatine, GLUT, glycogen and growth factor expression. - Brain health and concussion: The AIS document describes potential improvements in cognitive processing under energy stress and possible benefits following mild traumatic brain injury. 

## Retrieval inspection


In [63]:
def show_results(results):

    rows = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for text, metadata, distance in zip(
        documents,
        metadatas,
        distances,
    ):
        rows.append(
            {
                "source": metadata["source"],
                "title": metadata["title"],
                "topic": metadata["topic"],
                "distance": distance,
                "text": text,
            }
        )

    return pd.DataFrame(rows)

show_results(results)

,source,title,topic,distance,text
0,Australian Institute of Sport,Creatine,creatine,1.351352,seconds): Increased speed and reduced time to ...
1,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,1.400465,"running or swimming, that do not depend on the..."
2,Australian Institute of Sport,Creatine,creatine,1.407458,production. Brain creatine can be increased wi...
3,Australian Institute of Sport,Creatine,creatine,1.423925,during and/or at the end of endurance exercise...
4,Australian Institute of Sport,Creatine - Concerns and Considerations,creatine,1.425645,biomechanical disadvantage for some athletes. ...


## Topic filtering

In [64]:
def vector_search(
    query,
    topic=None,
    top_k=5,
):

    query_embedding = create_embedding(query)

    query_args = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
    }

    if topic is not None:
        query_args["where"] = {
            "topic": topic
        }

    return collection.query(**query_args)

## Testing
results = vector_search(
    query="Does creatine cause kidney damage",
    topic="creatine",
)

show_results(results)

,source,title,topic,distance,text
0,NIH NCCIH,Bodybuilding and Performance Enhancement Suppl...,creatine,0.596416,# What is creatine? Creatine is an amino acid ...
1,Australian Institute of Sport,Creatine - Concerns and Considerations,creatine,0.724515,# Are there any concerns or considerations? Th...
2,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,0.757236,"running or swimming, that do not depend on the..."
3,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,0.821816,a dietary supplement that it could have ergoge...
4,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,0.840080,# Creatine Creatine is one of the most thoroug...


## Claim Extraction

In [65]:
def extract_claims(user_input):

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
You extract medical claims from user messages.

Return each independent factual medical claim on a separate line.
Do not evaluate whether the claim is correct.
"""
            },
            {
                "role": "user",
                "content": user_input,
            },
        ],
        temperature=0,
    )

    output = (
        response
        .choices[0]
        .message
        .content
    )

    claims = [
        claim.strip("- ").strip()
        for claim in output.splitlines()
        if claim.strip()
    ]

    return claims

# Testing
user_input = """
    My friend said creatine damages your kidneys, causes permanent weight gain, and is a steroid.
"""
claims = extract_claims(user_input)
claims

['Creatine damages your kidneys.',
 'Creatine causes permanent weight gain.',
 'Creatine is a steroid.']

## Query Rewriting

In [66]:
def rewrite_query(claim):

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
Convert medical claims into concise evidence-search queries.

Return only the search query.
"""
            },
            {
                "role": "user",
                "content": claim,
            },
        ],
        temperature=0,
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

for claim in claims:
    query = rewrite_query(claim)
    print("Claim:", claim)
    print("Query:", query)
    print()

Claim: Creatine damages your kidneys.
Query: "Evidence on the effects of creatine supplementation on kidney function in healthy individuals."

Claim: Creatine causes permanent weight gain.
Query: "Does creatine supplementation lead to permanent weight gain in users?"

Claim: Creatine is a steroid.
Query: "Is creatine classified as a steroid in medical literature?"



## Evidence retrieval

In [67]:
all_evidence = []

for claim in claims:

    query = rewrite_query(claim)
    results = vector_search(
        query=query,
        top_k=5,
    )
    evidence_df = show_results(results)

    evidence_df["claim"] = claim
    evidence_df["query"] = query

    all_evidence.append(evidence_df)

In [68]:
# Combine
evidence_df = pd.concat(
    all_evidence,
    ignore_index=True,
)
evidence_df

,source,title,topic,distance,text,claim,query
0,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,0.606145,a dietary supplement that it could have ergoge...,Creatine damages your kidneys.,"""Evidence on the effects of creatine supplemen..."
1,Australian Institute of Sport,Creatine - Concerns and Considerations,creatine,0.620303,# Are there any concerns or considerations? Th...,Creatine damages your kidneys.,"""Evidence on the effects of creatine supplemen..."
2,NIH NCCIH,Bodybuilding and Performance Enhancement Suppl...,creatine,0.641907,# What is creatine? Creatine is an amino acid ...,Creatine damages your kidneys.,"""Evidence on the effects of creatine supplemen..."
3,Australian Institute of Sport,Creatine,creatine,0.687945,during and/or at the end of endurance exercise...,Creatine damages your kidneys.,"""Evidence on the effects of creatine supplemen..."
4,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,0.713658,# Creatine Creatine is one of the most thoroug...,Creatine damages your kidneys.,"""Evidence on the effects of creatine supplemen..."
5,Australian Institute of Sport,Creatine - Concerns and Considerations,creatine,0.541329,# Are there any concerns or considerations? Th...,Creatine causes permanent weight gain.,"""Does creatine supplementation lead to permane..."
6,Australian Institute of Sport,Creatine,creatine,0.623868,"discussion, including acute weight gain and ga...",Creatine causes permanent weight gain.,"""Does creatine supplementation lead to permane..."
7,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,0.645606,a dietary supplement that it could have ergoge...,Creatine causes permanent weight gain.,"""Does creatine supplementation lead to permane..."
8,NIH Office of Dietary Supplements,Exercise and Athletic Performance - Health Pro...,creatine,0.662818,"running or swimming, that do not depend on the...",Creatine causes permanent weight gain.,"""Does creatine supplementation lead to permane..."
9,Australian Institute of Sport,Creatine,creatine,0.704950,during and/or at the end of endurance exercise...,Creatine causes permanent weight gain.,"""Does creatine supplementation lead to permane..."


## Context Formatting

In [69]:
def format_context(results):

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    sections = []

    for i, (text, metadata) in enumerate(
        zip(documents, metadatas),
        start=1,
    ):

        section = f"""
SOURCE {i}

Title:
{metadata["title"]}

Organisation:
{metadata["source"]}

URL:
{metadata["url"]}

Evidence:
{text}
"""

        sections.append(section)

    return "\n".join(
        sections
    )

In [70]:
# Testing
results = vector_search(
    "creatine supplementation kidney function safety"
)
context = format_context(results)
print(context)


SOURCE 1

Title:
Bodybuilding and Performance Enhancement Supplements

Organisation:
NIH NCCIH

URL:
https://www.nccih.nih.gov/health/bodybuilding-and-performance-enhancement-supplements

Evidence:
# What is creatine? Creatine is an amino acid produced by the body and a popular dietary supplement. It may somewhat enhance the effects of exercise on strength, muscle mass, and endurance. But creatine can have some short-term side effects, and its long-term effects haven’t been well studied. # Is creatine safe? There have been reports that creatine may impair liver and kidney function. Creatine has also been linked to an increased risk of compartment syndrome, a condition where pressure builds in a muscle compartment and prevents blood flow. People at risk of kidney problems should check with their health care providers before using creatine and be carefully monitored while using it. There are no data documenting the safety of creatine in children or adolescents. The American Academy of P

## RAG Generation

In [ ]:
def generate_answer(
    question,
    context,
):

    system_prompt = """
You are an educational medical misinformation assistant aimed
primarily at young adults.

Your purpose is to explain medical evidence, not provide personalised
medical advice.

Rules:
- The reading Level of the response must be at year 8 reading level with a complexity under 15% according to SHLL.
    - To reduce the reading level:
        - Most words, if possible, should be less than 3 syllables
        - Use dot points for things that should be a list (if there are more than 3 things)
        - Sentences must be at less than 20 words long.
    - To reduce the complexity:
        - If you use a medical term, make sure to define it first.
        - Use simplest word of the same definition from the thesaurus.
        - Use common words where possible.
        - Define acronyms first before using it and minimise their use if possible.
    - Always use an active voice where possible.
- Use the retrieved evidence to support factual medical claims.
- Do not diagnose the user.
- Do not prescribe treatment.
- Clearly communicate uncertainty.
- Do not invent evidence.
- If the retrieved evidence is insufficient, say so.
- Cite evidence using [Source 1], [Source 2], etc.
- At the bottom, show the source links like so: [Source 1] --> link.
- Explain unfamiliar medical terminology using simple language.
- Always recommend seeing a medical practitioner and urge users to do their own further research.

Use this format:

Claim assessment:
Short answer:
What the evidence says:
Important context:
Evidence confidence:
Recommendation:
Links:
"""

    user_prompt = f"""
USER QUESTION

{question}


RETRIEVED EVIDENCE

{context}
"""

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],

        temperature=0.1,
    )

    return (
        response
        .choices[0]
        .message
        .content
    )

In [72]:
# Testing
answer = generate_answer(
    question=("Does creatine damage your kidneys?"),
    context=context,
)

print(answer)

**Claim assessment:** Does creatine damage your kidneys?

**Short answer:** There is no strong evidence that creatine damages healthy kidneys in otherwise healthy individuals.

**What the evidence says:**
- **Source 1** mentions that there have been reports suggesting creatine may impair kidney function and advises people at risk of kidney problems to consult healthcare providers before using creatine.
- **Source 2** and **Source 3** indicate that creatine is generally safe for healthy adults for both short-term and long-term use, with no consistent side effects other than weight gain due to water retention.
- **Source 5** from the Australian Institute of Sport states that there is no evidence of systematic serious adverse effects, including renal dysfunction, from creatine monohydrate supplementation.

**Important context:**
- While creatine is considered safe for healthy individuals, those with pre-existing kidney conditions should exercise caution. It’s important to stay well-hydrat

## RAG function

In [73]:
def rag_answer(
    user_input,
    top_k=5,
):

    results = vector_search(
        query=user_input,
        top_k=top_k,
    )

    context = format_context(results)

    answer = generate_answer(
        question=user_input,
        context=context,
    )

    return {
        "answer": answer,
        "results": results,
    }

In [74]:
# Testing
result = rag_answer("Does creatine improve muscle growth and strength?")
print(result["answer"])

**Claim Assessment:**
Does creatine improve muscle growth and strength?

**Short Answer:**
Yes, creatine supplementation is generally effective in improving muscle growth and strength, particularly when combined with resistance training.

**What the Evidence Says:**
- **Source 1:** Creatine supplementation combined with resistance exercise consistently improves resistance training outcomes, such as muscle strength, endurance, and muscle hypertrophy. It is particularly effective for high-intensity, short-duration activities.
- **Source 2:** Creatine supplementation can improve recovery from disuse atrophy and has multiple direct effects on muscle that could benefit athletic performance and adaptation to exercise training.
- **Source 3:** Short-term creatine supplementation significantly increases strength and power in both men and women, as shown in various studies involving resistance-trained individuals and athletes.
- **Source 4:** Creatine is one of the most studied supplements for 

## Claim-aware pipeline

In [75]:
def claim_rag_answer(
    user_input,
    top_k=5,
):

    claims = extract_claims(user_input)

    retrieved_results = []

    for claim in claims:

        query = rewrite_query(claim)

        results = vector_search(
            query=query,
            top_k=top_k,
        )

        retrieved_results.append(
            {
                "claim": claim,
                "query": query,
                "results": results,
            }
        )

    return retrieved_results

In [76]:
# Testing
result = claim_rag_answer(
    """
    Creatine causes kidney damage is only useful for professional bodybuilders.
    """
)

result

[{'claim': 'Creatine causes kidney damage.',
  'query': '"Evidence on whether creatine supplementation causes kidney damage in healthy individuals."',
  'results': {'ids': [['nih_creatine_safety_0',
     'ais_creatine_concerns_0',
     'nih_creatine_dosing_1',
     'nih_creatine_dosing_3',
     'nih_creatine_dosing_4']],
   'embeddings': None,
   'documents': [['# What is creatine? Creatine is an amino acid produced by the body and a popular dietary supplement. It may somewhat enhance the effects of exercise on strength, muscle mass, and endurance. But creatine can have some short-term side effects, and its long-term effects haven’t been well studied. # Is creatine safe? There have been reports that creatine may impair liver and kidney function. Creatine has also been linked to an increased risk of compartment syndrome, a condition where pressure builds in a muscle compartment and prevents blood flow. People at risk of kidney problems should check with their health care providers befor

## Experimentation / Evaluation

In [77]:
test_questions = [
    "Does creatine improve muscle strength?",
    "Does creatine help build muscle when combined with resistance training?",
    "Is creatine useful for long-distance running?",
    "Does creatine cause weight gain?",
    "Does creatine damage your kidneys?",
    "Is creatine monohydrate worse than creatine HCl?",
    "Do you need to load creatine for it to work?",
    "Is creatine safe for everyone?",
    "Can creatine supplements contain banned substances?",
    "Does creatine cause permanent water retention?"
]

for question in test_questions:

    print("=" * 80)
    print("QUESTION:")
    print(question)

    result = rag_answer(question)

    print("\nANSWER:")
    print(result["answer"])
    print()

QUESTION:
Does creatine improve muscle strength?

ANSWER:
**Claim Assessment:**
Does creatine improve muscle strength?

**Short Answer:**
Yes, creatine supplementation is consistently found to improve muscle strength, particularly when combined with resistance training.

**What the Evidence Says:**
Multiple sources indicate that creatine supplementation enhances muscle strength. According to the Australian Institute of Sport, creatine supplementation combined with resistance exercise improves resistance training outcomes such as muscle strength, endurance, and muscle hypertrophy [Source 1]. The NIH Office of Dietary Supplements also reports that studies have found short-term creatine supplementation significantly increases strength in activities like bench presses and power in cycling [Source 3]. Long-term supplementation helps with training adaptations and structured workloads over time [Source 4].

**Important Context:**
Creatine works by increasing phosphocreatine stores in muscles,

In [78]:
for question in test_questions:

    print("=" * 80)
    print("QUESTION:")
    print(question)

    result = rag_answer(question)

    print("\nANSWER:")
    print(result["answer"])
    print()

QUESTION:
Does creatine improve muscle strength?

ANSWER:
**Claim Assessment:**
Does creatine improve muscle strength?

**Short Answer:**
Yes, creatine supplementation is consistently found to improve muscle strength, particularly when combined with resistance exercise.

**What the Evidence Says:**
Multiple sources indicate that creatine supplementation enhances muscle strength. According to the Australian Institute of Sport, creatine supplementation combined with resistance exercise improves resistance training outcomes such as muscle strength, endurance, and muscle hypertrophy [Source 1]. The NIH Office of Dietary Supplements reports that studies have found short-term creatine supplementation significantly increases strength in activities like bench presses and power in cycling [Source 3]. Long-term supplementation also aids in training adaptations and increased workloads over time [Source 4].

**Important Context:**
Creatine works by increasing stores of phosphocreatine in muscles, 